In [ ]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
)
from llama_parse import LlamaParse

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
# Each will retrieve the top-2 most similar nodes
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import get_response_synthesizer
from llama_index.core.evaluation import (
    EvaluationResult,
    RelevancyEvaluator,
    CorrectnessEvaluator,
)
from llama_index.core import Response
import pandas as pd
import asyncio
from llama_index.core.llama_dataset import (
    LabelledRagDataExample,
)
from types import SimpleNamespace
import json

# NESTED ASYNCIO LOOP NEEDED TO RUN ASYNC IN A NOTEBOOK
import nest_asyncio

nest_asyncio.apply()

# Settings model
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-zh-v1.5")
Settings.llm = Ollama(model="llama3.1:latest", request_timeout=60.0, temperature=0.3)

# create the parser
parser = LlamaParse(result_type="markdown")
file_extractor = {".pdf": parser}

# load the documents
documents = SimpleDirectoryReader("data", file_extractor=file_extractor).load_data()

PERSIST_DATASET_FILE = "./rag_dataset.json"
# load the dataset
with open(PERSIST_DATASET_FILE, "r") as f:
    # 將字典轉換為支持屬性訪問的結構
    rag_dataset = json.load(f, object_hook=lambda d: SimpleNamespace(**d))


def evaluate_response(chunk_size):
    # chunk
    text_splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=20)
    # create the index
    index = VectorStoreIndex.from_documents(documents, transformations=[text_splitter])

    # configure retriever
    vector_retriever = index.as_retriever(similarity_top_k=2, verbose=True)
    bm25_retriever = BM25Retriever.from_defaults(
        docstore=index.docstore, similarity_top_k=2
    )
    retriever = QueryFusionRetriever(
        [vector_retriever, bm25_retriever],
        similarity_top_k=3,
        num_queries=1,  # set this to 1 to disable query generation
        mode="reciprocal_rerank",
        use_async=True,
        verbose=True,
    )

    # configure response synthesizer
    response_synthesizer = get_response_synthesizer()

    # assemble query engine
    query_engine = RetrieverQueryEngine(
        retriever=retriever,
        response_synthesizer=response_synthesizer,
    )

    # 執行查詢引擎並評估結果
    run_evaluate_chat_engine(query_engine, rag_dataset.examples)

# 累積顯示評估結果的函數
def add_eval_df(
    response: Response,
    dataset_example: LabelledRagDataExample,
    relevancy_eval_result: EvaluationResult,
    correctness_eval_result: EvaluationResult,
) -> None:
    if not response.source_nodes:
        print("no response!")
        return
    eval_row = pd.DataFrame(
        [
            {
                "Query": dataset_example.query,
                "Response": str(response),
                "Reference Answer": dataset_example.reference_answer,
                "Source": response.source_nodes[0].node.text[:1000] + "...",
                "Relevancy Eval Result": f"{'Pass' if relevancy_eval_result.passing else 'Fail'} \nscore: {relevancy_eval_result.score}",
                "Relevancy Reasoning": relevancy_eval_result.feedback,
                "Correctness Eval Result": f"{'Pass' if correctness_eval_result.passing else 'Fail'} \n\nscore: {correctness_eval_result.score}",
                "Correctness Reasoning": correctness_eval_result.feedback,
            }
        ]
    )
    global eval_results_df
    eval_results_df = pd.concat([eval_results_df, eval_row], ignore_index=True)


# 評估查詢引擎並顯示最終結果
async def evaluate_chat_engine(chat_engine, dataset_examples: LabelledRagDataExample):
    import time

    dataset_examples = dataset_examples[:3]

    # 定義評估器
    relevancy_evaluator = RelevancyEvaluator()
    correctness_evaluator = CorrectnessEvaluator()
    relevancy_total_correct = 0
    correctness_total_correct = 0
    correctness_total_score = 0
    for dataset_example in dataset_examples:
        start_time = time.time()
        response = chat_engine.query(dataset_example.query)
        process_time = time.time() - start_time

        relevancy_eval_result = relevancy_evaluator.evaluate_response(
            query=dataset_example.query, response=response
        )
        correctness_eval_result = correctness_evaluator.evaluate_response(
            query=dataset_example.query,
            response=response,
            reference=dataset_example.reference_answer,
        )

        # 累積每個查詢的評估結果
        add_eval_df(
            response, dataset_example, relevancy_eval_result, correctness_eval_result
        )

        if relevancy_eval_result.passing:
            relevancy_total_correct += 1
        if correctness_eval_result.passing:
            correctness_total_correct += 1
            correctness_total_score += correctness_eval_result.score

    return (
        relevancy_total_correct,
        correctness_total_correct,
        correctness_total_score,
        len(dataset_examples),
        process_time,
    )


# 執行評估查詢並顯示最終累積結果
def run_evaluate_chat_engine(chat_engine, dataset_examples):
    (
        relevancy_total_correct,
        correctness_total_correct,
        correctness_total_score,
        total_questions,
        process_time,
    ) = asyncio.run(evaluate_chat_engine(chat_engine, dataset_examples))
    global eval_results_df

    styled_df = eval_results_df.style.set_properties(
        **{"white-space": "pre-wrap"},
    )

    display(styled_df)
    # 顯示總結結果
    print(
        f"Total Relevancy correct: {relevancy_total_correct} out of {total_questions}, score: {relevancy_total_correct/total_questions}"
    )
    print(
        f"Total Correctness correct: {correctness_total_correct} out of {total_questions}, score: {correctness_total_score/total_questions}"
    )
    print(f"Total response process time: {process_time} seconds")

for chunk_size in [1024,2048]:
    # 累積評估結果的 DataFrame
    print(f"Chunk size: {chunk_size}")
    eval_results_df = pd.DataFrame()
    evaluate_response(chunk_size)
    print("===============================================")

resource module not available on Windows
Started parsing the file under job_id efbb493a-751d-49c5-800d-187c6d9668cc
Started parsing the file under job_id d27d94ad-e51f-42bb-971d-02a8ed0b0e39
Started parsing the file under job_id e8b51e0a-8811-488f-9581-a2d68a8f5cdd
Started parsing the file under job_id e3ceb27d-2150-425d-a4f3-ab62e00293a4
Started parsing the file under job_id 30bee8f5-2e19-4bce-8370-2b0764c9699e
Started parsing the file under job_id 2fcd9d26-6ad6-4451-a4e0-cfe851a9ff62
Chunk size: 1024


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning
0,什麼是學餐資訊？,這是一份關於未來大學的學生餐廳資訊的手冊。它介紹了不同類型的餐廳，例如惜食餐廳、有機農場餐廳、健康小食堂等，每個餐廳都強調推動某些SDGs目標的概念，並提供相關菜單和服務。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,# 學餐資訊 為倡導SDGs的概念，校內的學生餐廳也是採用其中幾個目標的概念來建立： # 22 ZERO HUNGER # 1. 每日學餐（強調SDGs目標1：零飢餓 Zero Hunger） - (1) 每日惜食餐廳 - 提供當日生鮮剩餘食材製作的健康餐點，減少食物浪費。 - 招牌菜：惜食蔬菜濃湯、健康全麥三明治。 - (2) 有機農場餐廳 - 使用校園農業計劃種植的有機蔬菜，提供新鮮、健康的餐點。 - 招牌菜：有機沙拉碗、田園蔬菜披薩。 - (3) 健康小食堂 - 專為有需要的學生提供免費或低價的營養餐點。 - 招牌菜：營養便當、五穀雜糧飯。 # 12 RESPONSIBLE CONSUMPTION AND PRODUCTION # 2. Eco Eats學餐（強調SDGs目標12：負責任的消費和生產） - (1) 綠色餐廳 - 提供本地採購的食材，減少食物碳足跡。 - 招牌菜：本地風味烤雞、鮮果沙拉。 - (2) 廢物再利用餐廳 - 使用食材的每一部分，推動零廢棄理念。 - 招牌菜：蔬菜莖葉炒飯、柑橘皮蜜餞。 - (3) 環保快餐店 - 所有餐具均為可再生或可降解材料，提倡環保用餐。 - 招牌菜：竹纖維碗拉麵、紙漿盒三明治。 # 3 GOOD HEALTH AND WELL-BEING # 3. 健康學餐（強調SDGs目標3：良好健康與福祉） - (1) 運動員餐廳 - 提供高蛋白、低脂肪的餐點，適合運動後補充能量。 - 招牌菜：健身沙拉、雞胸肉餐盒。 - (2) 心靈健康餐廳 - 提供低糖、低鹽的餐點，支持心理健康。 - 招牌菜：穀物水果碗、香菇豆腐湯。 - (3) 健康果汁吧 - 提供新鮮水果和蔬菜汁，增強免疫力。 - 招牌菜：綠色活力汁、莓果奶昔。...,Pass score: 1.0,YES.,Pass score: 4.0,"The generated answer is relevant to the user query, as it discusses a type of information related to student dining in schools. However, it contains some inaccuracies and focuses on a specific university's student restaurants, whereas the reference answer provides a more general definition of ""學餐資訊"". The generated answer also introduces new concepts (SDGs) that are not present in the reference answer, which might indicate a slight deviation from the expected response. Nevertheless, the overall content is still relevant and informative, hence the score of 4.0."
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,校內的學生餐廳採用零飢餓（Zero Hunger）、負責任的消費和生產（Responsible Consumption and Production）、良好健康與福祉（Good Health and Well-being）等SDGs目標的概念來建立。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,YES,Pass score: 4.0,"The generated answer is fully relevant to the user query, and it accurately lists three SDGs targets that are adopted by the student cafeteria. The only difference between the generated answer and the reference answer is that the former uses a more concise format for listing the targets, but this does not affect the correctness of the information provided."
2,什麼是 Green Garden 學餐？,社區農園餐廳。,社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,"YES. The response ""社區農園餐廳"" is in line with the context information provided, which describes Green Garden 學餐 as a community garden restaurant that utilizes school community garden's vegetables to provide fresh and healthy meals.",Pass score: 5.0,"The generated answer is identical to the reference answer, which suggests that it is fully correct and relevant to the user query. There are no mistak

Total Relevancy correct: 3 out of 3, score: 1.0
Total Correctness correct: 3 out of 3, score: 4.333333333333333
Chunk size: 2048


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning
0,什麼是學餐資訊？,這是一份關於未來大學校園學生餐廳的資訊手冊，內容包括各個餐廳的特色、招牌菜品等信息。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,# 學餐資訊 為倡導SDGs的概念，校內的學生餐廳也是採用其中幾個目標的概念來建立： # 22 ZERO HUNGER # 1. 每日學餐（強調SDGs目標1：零飢餓 Zero Hunger） - (1) 每日惜食餐廳 - 提供當日生鮮剩餘食材製作的健康餐點，減少食物浪費。 - 招牌菜：惜食蔬菜濃湯、健康全麥三明治。 - (2) 有機農場餐廳 - 使用校園農業計劃種植的有機蔬菜，提供新鮮、健康的餐點。 - 招牌菜：有機沙拉碗、田園蔬菜披薩。 - (3) 健康小食堂 - 專為有需要的學生提供免費或低價的營養餐點。 - 招牌菜：營養便當、五穀雜糧飯。 # 12 RESPONSIBLE CONSUMPTION AND PRODUCTION # 2. Eco Eats學餐（強調SDGs目標12：負責任的消費和生產） - (1) 綠色餐廳 - 提供本地採購的食材，減少食物碳足跡。 - 招牌菜：本地風味烤雞、鮮果沙拉。 - (2) 廢物再利用餐廳 - 使用食材的每一部分，推動零廢棄理念。 - 招牌菜：蔬菜莖葉炒飯、柑橘皮蜜餞。 - (3) 環保快餐店 - 所有餐具均為可再生或可降解材料，提倡環保用餐。 - 招牌菜：竹纖維碗拉麵、紙漿盒三明治。 # 3 GOOD HEALTH AND WELL-BEING # 3. 健康學餐（強調SDGs目標3：良好健康與福祉） - (1) 運動員餐廳 - 提供高蛋白、低脂肪的餐點，適合運動後補充能量。 - 招牌菜：健身沙拉、雞胸肉餐盒。 - (2) 心靈健康餐廳 - 提供低糖、低鹽的餐點，支持心理健康。 - 招牌菜：穀物水果碗、香菇豆腐湯。 - (3) 健康果汁吧 - 提供新鮮水果和蔬菜汁，增強免疫力。 - 招牌菜：綠色活力汁、莓果奶昔。...,Pass score: 1.0,YES.,Pass score: 4.0,"The generated answer is relevant to the user query, as it provides information about school cafeteria food, which aligns with the concept of ""學餐資訊"". However, it contains some inaccuracies, such as mentioning a ""未來大學校園"" (future university campus) instead of the current context of school cafeterias. The answer also strays from the specific focus on food information, but still provides some relevant details about cafeteria features and menu items. Overall, it demonstrates some understanding of the concept, but with room for improvement in accuracy and relevance to the user query."
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,校內的學生餐廳採用了SDGs目標1：零飢餓、目標12：負責任的消費和生產以及目標3：良好健康與福祉的概念來建立。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,YES,Pass score: 4.0,"The generated answer is fully relevant to the user query, and it accurately lists all three SDGs targets mentioned in the reference answer. The formatting is also consistent with the reference answer, making it easy to compare and contrast. The only minor difference is that the generated answer uses a comma to separate the targets, while the reference answer uses line breaks. However, this does not affect the overall correctness or relevance of the answer."
2,什麼是 Green Garden 學餐？,社區農園餐廳提供新鮮健康的餐點，利用校園社區農園的蔬菜。招牌菜有社區農園沙拉和蔬菜焗飯。,社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,"YES. The response for the query ""什麼是 Green Garden 學餐？"" is in line with the context information provided, which describes a community garden restaurant that serves fresh and healthy meals using vegetables from the school's community garden. The menu items mentioned in the response (社區農園沙拉 and 蔬菜焗飯) are also consistent with the description of the Gree

Total Relevancy correct: 3 out of 3, score: 1.0
Total Correctness correct: 3 out of 3, score: 4.0
